# Clusters K-Means esférico sobre el train completo

Calcula los prototipos (centroides) de las celdas de Voronoi esféricas usando **todo el train**
de `DATA_URL`, no una muestra. A diferencia de `celdas_voronoi.ipynb` (que ajustó sobre la muestra de 20k
balanceada por país), los centroides reflejan la distribución real del dataset.

1. Lee solo `latitude`, `longitude`, `image_id` y `panoid` de todos los `train-*.parquet`.
2. **Excluye la validación global**: filas cuyo `panoid` o `image_id` figura en
   `datasets/samples/data/validation/validation_ids.parquet`.
3. **Un punto por panorama**: las vistas de un mismo `panoid` comparten coordenada; sin deduplicar,
   cada locación pesaría ~4 veces.
4. Ajusta K-Means esférico (mismas funciones que `celdas_voronoi.ipynb`) con `K = 70`.
5. Guarda en `assets/`:
   - `prototipos_train-completo_k70.csv` con el mismo esquema que `prototipos.csv` (`ind_cluster, lat, lon`),
     así lo pueden usar directamente `create_samples_celdas.ipynb` y los clasificadores por celda.
   - `prototipos_train-completo_k70.json` con la metadata de la corrida.

No se sobrescriben archivos existentes.

In [1]:
from datetime import datetime, timezone
from pathlib import Path
import hashlib
import json
import os

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from IPython.display import display
from tqdm.auto import tqdm

# Parámetros editables.
K = 70
N_INIT = 5
MAX_ITER = 100
TOL = 1e-7
SEMILLA = 42
PATRON_ARCHIVOS = "**/train-*.parquet"
VALIDACION_IDS = "datasets/samples/data/validation/validation_ids.parquet"
VALIDACION_CONFIG = "datasets/samples/data/validation/validation_config.json"
PROTOTIPOS_ANTERIORES = "src/notebooks/preprocessing/assets/prototipos.csv"  # Solo para comparar.
SALIDA_CSV = f"src/notebooks/preprocessing/assets/prototipos_train-completo_k{K}.csv"
SALIDA_JSON = f"src/notebooks/preprocessing/assets/prototipos_train-completo_k{K}.json"
COL_LAT, COL_LON = "latitude", "longitude"
R_TIERRA = 6371.0

c:\Users\jurad\Documents\facultad\rrnn2\tp-final\geo-project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Localizar el dataset

`DATA_URL` se toma del entorno si ya está definida; de lo contrario se lee del `.env`. Las rutas
relativas se resuelven respecto de la carpeta del `.env` (raíz del proyecto).

In [2]:
env_path = next(
    (carpeta / ".env" for carpeta in (Path.cwd(), *Path.cwd().parents)
     if (carpeta / ".env").is_file()),
    None,
)
if env_path is None:
    raise FileNotFoundError("No se encontró el .env del proyecto desde el directorio actual.")

valor_env = ""
for linea in env_path.read_text(encoding="utf-8-sig").splitlines():
    clave, separador, valor = linea.partition("=")
    if separador and clave.strip() == "DATA_URL":
        valor_env = valor.strip().strip("\"'")

ruta_configurada = os.environ.get("DATA_URL", valor_env).strip()
if not ruta_configurada:
    raise ValueError("Configurá DATA_URL en el .env con la carpeta del dataset local.")

raiz_proyecto = env_path.parent


def resolver(ruta):
    ruta = Path(ruta).expanduser()
    return (ruta if ruta.is_absolute() else raiz_proyecto / ruta).resolve()


ruta_datos = resolver(ruta_configurada)
if not ruta_datos.is_dir():
    raise NotADirectoryError(f"DATA_URL apunta a una carpeta inexistente: {ruta_datos}")
archivos = sorted(p for p in ruta_datos.glob(PATRON_ARCHIVOS) if p.is_file())
if not archivos:
    raise FileNotFoundError(f"No se encontraron Parquet con {PATRON_ARCHIVOS!r} en {ruta_datos}")

ruta_csv, ruta_json = resolver(SALIDA_CSV), resolver(SALIDA_JSON)
existentes = [p for p in (ruta_csv, ruta_json) if p.exists()]
if existentes:
    raise FileExistsError(f"La salida ya existe: {existentes}. No se sobrescribe.")

print(f"Dataset: {ruta_datos}")
print(f"Archivos de entrada: {len(archivos):,}")
print(f"Salidas: {ruta_csv.name}, {ruta_json.name}")

Dataset: C:\Users\jurad\Documents\facultad\rrnn2\tp-final\world-streetview-500k\data
Archivos de entrada: 393
Salidas: prototipos_train-completo_k70.csv, prototipos_train-completo_k70.json


## Validación global a excluir

Se leen solo los IDs. La huella SHA-256 se calcula igual que en `create_validation_split.ipynb`
(pares `[image_id, panoid]` ordenados) y se compara con la de `validation_config.json`.

In [3]:
ruta_val_ids = resolver(VALIDACION_IDS)
validacion = pq.read_table(ruta_val_ids, columns=["image_id", "panoid"]).to_pandas()
for columna in ("image_id", "panoid"):
    validacion[columna] = validacion[columna].astype("string").str.strip().replace("", pd.NA)
if validacion.empty or validacion.isna().any().any():
    raise ValueError("La validación global debe tener filas e identificadores no vacíos.")

ids_ordenados = validacion.sort_values(["image_id", "panoid"]).to_numpy().tolist()
huella_val = hashlib.sha256(
    json.dumps(ids_ordenados, ensure_ascii=False, separators=(",", ":")).encode("utf-8")
).hexdigest()
huella_config = json.loads(resolver(VALIDACION_CONFIG).read_text(encoding="utf-8")).get("sha256_ids")
if huella_config is not None and huella_config != huella_val:
    raise ValueError("Los IDs de validación no coinciden con validation_config.json.")

ids_val, panos_val = set(validacion["image_id"]), set(validacion["panoid"])
print(f"Validación global: {len(validacion):,} imágenes, {len(panos_val):,} panoramas. Huella OK.")

Validación global: 5,332 imágenes, 1,333 panoramas. Huella OK.


## 1. Coordenadas del train completo

Se leen solo cuatro columnas por archivo (sin imágenes). Se descartan filas reservadas para validación,
con IDs vacíos o con coordenadas inválidas, y después se deja una fila por `panoid`.

In [4]:
def coordenadas_validas(lat, lon):
    return np.isfinite(lat) & np.isfinite(lon) & (np.abs(lat) <= 90) & (np.abs(lon) <= 180)


columnas = ["image_id", "panoid", COL_LAT, COL_LON]
partes, inventario = [], []
for archivo in tqdm(archivos, desc="Leyendo coordenadas"):
    tabla = pq.read_table(archivo, columns=columnas).to_pandas()
    inventario.append({"archivo": archivo.relative_to(ruta_datos).as_posix(), "filas": len(tabla)})
    partes.append(tabla)
filas = pd.concat(partes, ignore_index=True)
del partes

for columna in ("image_id", "panoid"):
    filas[columna] = filas[columna].astype("string").str.strip().replace("", pd.NA)
filas["lat"] = pd.to_numeric(filas.pop(COL_LAT), errors="coerce")
filas["lon"] = pd.to_numeric(filas.pop(COL_LON), errors="coerce")

ids_invalidos = (filas["image_id"].isna() | filas["panoid"].isna()).to_numpy()
coords_invalidas = ~coordenadas_validas(filas["lat"].to_numpy(float), filas["lon"].to_numpy(float))
reservadas = (filas["image_id"].isin(ids_val) | filas["panoid"].isin(panos_val)).to_numpy()
elegibles = ~ids_invalidos & ~coords_invalidas & ~reservadas

coords = filas.loc[elegibles].drop_duplicates("panoid", keep="first").reset_index(drop=True)

# Las vistas de un panorama deberían compartir coordenada; se informa la máxima discrepancia.
por_pano = filas.loc[elegibles].groupby("panoid")[["lat", "lon"]]
dispersion = (por_pano.max() - por_pano.min()).max()

conteos = {
    "filas_leidas": int(len(filas)),
    "ids_invalidos": int(ids_invalidos.sum()),
    "coordenadas_invalidas": int((coords_invalidas & ~ids_invalidos).sum()),
    "excluidas_validacion": int((reservadas & ~ids_invalidos & ~coords_invalidas).sum()),
    "filas_elegibles": int(elegibles.sum()),
    "panoramas_usados": int(len(coords)),
}
for clave, valor in conteos.items():
    print(f"{clave:>24}: {valor:,}")
print(f"Máx. diferencia de coordenadas dentro de un panoid: lat {dispersion.lat:.2e}°, lon {dispersion.lon:.2e}°")
del filas

Leyendo coordenadas: 100%|██████████| 393/393 [00:01<00:00, 231.73it/s]


            filas_leidas: 1,999,060
           ids_invalidos: 0
   coordenadas_invalidas: 0
    excluidas_validacion: 5,332
         filas_elegibles: 1,993,728
        panoramas_usados: 498,432
Máx. diferencia de coordenadas dentro de un panoid: lat 0.00e+00°, lon 0.00e+00°


## 2. K-Means esférico

Cada punto se convierte a un vector unitario en $\mathbb{R}^3$. La asignación es por mayor producto punto
(menor distancia angular) y el centroide es la media de los vectores reproyectada a la esfera.
Inicialización tipo k-means++ con distancia angular. Mismo algoritmo que `celdas_voronoi.ipynb`.

In [5]:
def to_xyz(lat, lon):
    lat, lon = np.radians(lat), np.radians(lon)
    return np.column_stack([np.cos(lat) * np.cos(lon), np.cos(lat) * np.sin(lon), np.sin(lat)])


def to_latlon(xyz):
    xyz = xyz / np.linalg.norm(xyz, axis=1, keepdims=True)
    return np.degrees(np.arcsin(np.clip(xyz[:, 2], -1, 1))), np.degrees(np.arctan2(xyz[:, 1], xyz[:, 0]))


def asignar(X, C, chunk=200_000):
    labels = np.empty(len(X), dtype=np.int32)
    sim = np.empty(len(X))
    for i in range(0, len(X), chunk):
        s = X[i:i + chunk] @ C.T
        labels[i:i + chunk] = s.argmax(axis=1)
        sim[i:i + chunk] = s.max(axis=1)
    return labels, sim


def init_pp(X, k, rng):
    C = [X[rng.integers(len(X))]]
    d = 1 - np.clip(X @ C[0], -1, 1)
    for _ in range(1, k):
        C.append(X[rng.choice(len(X), p=d / d.sum())])
        d = np.minimum(d, 1 - np.clip(X @ C[-1], -1, 1))
    return np.array(C)


def spherical_kmeans(X, k, n_init=5, max_iter=100, tol=1e-7, seed=0):
    rng = np.random.default_rng(seed)
    best = None
    for intento in range(n_init):
        C = init_pp(X, k, rng)
        prev = -np.inf
        for it in range(max_iter):
            labels, sim = asignar(X, C)
            for j in range(k):
                m = labels == j
                if m.any():
                    v = X[m].sum(axis=0)
                    C[j] = v / np.linalg.norm(v)
                else:  # cluster vacío: reubicar en el punto peor representado
                    C[j] = X[sim.argmin()]
            score = sim.sum()
            if score - prev < tol * len(X):
                break
            prev = score
        labels, sim = asignar(X, C)
        print(f"intento {intento + 1}/{n_init}: {it + 1} iteraciones, similitud media {sim.mean():.6f}")
        if best is None or sim.sum() > best[0]:
            best = (sim.sum(), C.copy())
    return best[1]


X = to_xyz(coords.lat.to_numpy(), coords.lon.to_numpy())
centroides = spherical_kmeans(X, K, N_INIT, MAX_ITER, TOL, SEMILLA)
labels, sim = asignar(X, centroides)
dist_km = np.arccos(np.clip(sim, -1, 1)) * R_TIERRA
n_por_celda = np.bincount(labels, minlength=K)
print(f"celdas no vacías: {(n_por_celda > 0).sum()} de {K}")
print(f"distancia al centroide (km): media {dist_km.mean():.0f}, mediana {np.median(dist_km):.0f}, máx {dist_km.max():.0f}")

intento 1/5: 20 iteraciones, similitud media 0.998087
intento 2/5: 38 iteraciones, similitud media 0.997991
intento 3/5: 39 iteraciones, similitud media 0.998090
intento 4/5: 59 iteraciones, similitud media 0.998065
intento 5/5: 44 iteraciones, similitud media 0.998061
celdas no vacías: 70 de 70
distancia al centroide (km): media 344, mediana 319, máx 2823


In [6]:
cent_lat, cent_lon = to_latlon(centroides)
dist_por_celda = pd.Series(dist_km).groupby(labels)
tabla_cent = pd.DataFrame({
    "lat": cent_lat, "lon": cent_lon, "n_panoramas": n_por_celda,
    "dist_media_km": dist_por_celda.mean().reindex(range(K)),
    "dist_max_km": dist_por_celda.max().reindex(range(K)),
}).rename_axis("ind_cluster").round(3)
display(tabla_cent.describe().round(1))
tabla_cent.sort_values("n_panoramas", ascending=False).head(10)

,lat,lon,n_panoramas,dist_media_km,dist_max_km
count,70.0,70.0,70.0,70.0,70.0
mean,15.2,8.6,7120.5,378.9,1215.1
std,30.8,87.7,5183.2,126.1,591.9
min,-44.4,-156.7,33.0,157.2,351.0
25%,-10.3,-69.2,2496.2,297.1,811.0
50%,20.4,10.1,6475.0,354.7,1034.7
75%,41.0,81.6,11129.5,428.9,1394.6
max,62.9,173.9,19818.0,806.9,2823.2


,lat,lon,n_panoramas,dist_media_km,dist_max_km
ind_cluster,,,,,
65,46.174,1.934,19818,275.901,731.354
30,41.822,-87.179,18422,309.404,918.495
63,15.492,103.182,17647,387.768,1215.381
49,32.698,-86.062,17478,346.413,1014.161
2,42.522,-76.532,16277,301.756,1536.906
5,50.647,7.928,15276,272.573,522.441
69,36.695,-80.857,15074,273.487,505.105
19,23.906,82.801,13845,319.211,669.054
13,17.947,77.036,13692,288.947,674.037


### Mapa del diagrama de Voronoi

Puntos coloreados por celda, centroides nuevos (×) y, como referencia, los de `prototipos.csv` (○).

In [7]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy.spatial import SphericalVoronoi

MAX_PUNTOS_MAPA = 30_000  # solo afecta al dibujo


def arco(a, b, n=24):
    omega = np.arccos(np.clip(a @ b, -1, 1))
    if omega < 1e-9:
        return np.array([a, b])
    t = np.linspace(0, 1, n)[:, None]
    return (np.sin((1 - t) * omega) * a + np.sin(t * omega) * b) / np.sin(omega)


def aristas_voronoi(C):
    sv = SphericalVoronoi(C, radius=1.0)
    sv.sort_vertices_of_regions()
    vistas, lats, lons = set(), [], []
    for region in sv.regions:
        for i in range(len(region)):
            u, v = region[i], region[(i + 1) % len(region)]
            clave = (min(u, v), max(u, v))
            if clave in vistas:
                continue
            vistas.add(clave)
            la, lo = to_latlon(arco(sv.vertices[u], sv.vertices[v]))
            la, lo = list(la), list(lo)
            # cortar la línea si cruza el antimeridiano
            for j in reversed(np.where(np.abs(np.diff(lo)) > 180)[0]):
                la.insert(j + 1, None); lo.insert(j + 1, None)
            lats += la + [None]; lons += lo + [None]
    return lats, lons


anteriores = pd.read_csv(resolver(PROTOTIPOS_ANTERIORES))
e_lat, e_lon = aristas_voronoi(centroides)
muestra = np.random.default_rng(SEMILLA).choice(len(coords), min(MAX_PUNTOS_MAPA, len(coords)), replace=False)

fig = go.Figure()
fig.add_trace(go.Scattergeo(lat=coords.lat.to_numpy()[muestra], lon=coords.lon.to_numpy()[muestra], mode="markers",
                            marker=dict(size=3, color=labels[muestra] % 10, colorscale="Turbo", opacity=0.5),
                            name="panoramas", hoverinfo="skip"))
fig.add_trace(go.Scattergeo(lat=e_lat, lon=e_lon, mode="lines", line=dict(width=1.2, color="#d62728"), name="Voronoi"))
fig.add_trace(go.Scattergeo(lat=anteriores.lat, lon=anteriores.lon, mode="markers",
                            marker=dict(size=7, color="rgba(0,0,0,0)", line=dict(width=1.5, color="#555555")),
                            name="prototipos.csv (anterior)"))
fig.add_trace(go.Scattergeo(lat=cent_lat, lon=cent_lon, mode="markers+text", text=[str(i) for i in range(K)],
                            textposition="top center", textfont=dict(size=8),
                            marker=dict(size=8, color="black", symbol="x"), name="centroides nuevos"))
fig.update_geos(projection_type="natural earth", showland=True, landcolor="#f1ede4", showocean=True,
                oceancolor="#dfeaf2", showcountries=True, countrycolor="#bbbbbb")
fig.update_layout(title=f"Celdas de Voronoi esféricas, K={K} (train completo)", height=650,
                  margin=dict(l=0, r=0, t=50, b=0), legend=dict(x=0.01, y=0.01))
fig.show()

In [8]:
fig = make_subplots(rows=1, cols=2, subplot_titles=("Histograma de panoramas por celda", "Panoramas por celda (ordenado)"))
fig.add_trace(go.Histogram(x=n_por_celda, nbinsx=30, marker_color="#1f77b4"), row=1, col=1)
orden = np.argsort(-n_por_celda)
fig.add_trace(go.Bar(x=orden.astype(str), y=n_por_celda[orden], marker_color="#1f77b4"), row=1, col=2)
fig.update_xaxes(title_text="panoramas", row=1, col=1)
fig.update_yaxes(title_text="celdas", row=1, col=1)
fig.update_xaxes(title_text="ind_cluster", type="category", row=1, col=2)
fig.update_yaxes(title_text="panoramas", row=1, col=2)
fig.update_layout(height=420, showlegend=False)
fig.show()

## 3. Guardar prototipos y metadata

El CSV tiene exactamente las columnas `ind_cluster, lat, lon`, como `prototipos.csv`. El JSON guarda los
parámetros, los conteos, las huellas de la validación excluida y del CSV, y estadísticas por celda
para poder reproducir y comparar la corrida.

In [9]:
prototipos = pd.DataFrame({"ind_cluster": range(K), "lat": cent_lat, "lon": cent_lon})
ruta_csv.parent.mkdir(parents=True, exist_ok=True)
with open(ruta_csv, "x", newline="", encoding="utf-8") as f:
    prototipos.to_csv(f, index=False)

metadata = {
    "tipo": "prototipos_kmeans_esferico",
    "fecha_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "notebook": "src/notebooks/preprocessing/clusters_train_completo.ipynb",
    "archivo_prototipos": ruta_csv.name,
    "sha256_prototipos": hashlib.sha256(ruta_csv.read_bytes()).hexdigest(),
    "parametros": {"k": K, "n_init": N_INIT, "max_iter": MAX_ITER, "tol": TOL, "semilla": SEMILLA,
                   "distancia": "angular (producto punto sobre la esfera unitaria)",
                   "init": "k-means++ con distancia angular"},
    "fuente": {"patron_archivos": PATRON_ARCHIVOS, "archivos": len(archivos),
               "filas_por_archivo": inventario},
    "deduplicacion": "un punto por panoid (primera fila elegible)",
    "validacion_excluida": {"archivo": VALIDACION_IDS, "columnas_exclusion": ["image_id", "panoid"],
                            "imagenes": int(len(validacion)), "panoramas": len(panos_val),
                            "sha256_ids": huella_val},
    "conteos": conteos,
    "resultado": {
        "celdas_no_vacias": int((n_por_celda > 0).sum()),
        "similitud_media": float(sim.mean()),
        "dist_centroide_km": {"media": float(dist_km.mean()), "mediana": float(np.median(dist_km)),
                              "max": float(dist_km.max())},
        "panoramas_por_celda": {"min": int(n_por_celda.min()), "max": int(n_por_celda.max()),
                                "media": float(n_por_celda.mean())},
        "por_celda": [
            {"ind_cluster": int(i), "n_panoramas": int(r.n_panoramas),
             "dist_media_km": float(r.dist_media_km), "dist_max_km": float(r.dist_max_km)}
            for i, r in tabla_cent.iterrows()
        ],
    },
}
with open(ruta_json, "x", encoding="utf-8") as f:
    json.dump(metadata, f, ensure_ascii=False, indent=2)

print(f"{K} prototipos guardados en {ruta_csv}")
print(f"Metadata guardada en {ruta_json}")

70 prototipos guardados en C:\Users\jurad\Documents\facultad\rrnn2\tp-final\geo-project\src\notebooks\preprocessing\assets\prototipos_train-completo_k70.csv
Metadata guardada en C:\Users\jurad\Documents\facultad\rrnn2\tp-final\geo-project\src\notebooks\preprocessing\assets\prototipos_train-completo_k70.json


## 4. Verificación

Relee el CSV como lo hace `create_samples_celdas.ipynb` y comprueba que reasignar los panoramas con los
prototipos guardados da las mismas celdas.

In [10]:
releido = pd.read_csv(ruta_csv)
assert list(releido.columns) == ["ind_cluster", "lat", "lon"]
assert (releido["ind_cluster"].to_numpy() == np.arange(K)).all()
assert coordenadas_validas(releido.lat.to_numpy(float), releido.lon.to_numpy(float)).all()
labels_releidos, _ = asignar(X, to_xyz(releido.lat.to_numpy(float), releido.lon.to_numpy(float)))
coincidencia = (labels_releidos == labels).mean()
print(f"Coincidencia de asignaciones tras releer el CSV: {coincidencia:.6%}")
assert coincidencia > 0.9999
print(f"Para usarlos: ARCHIVO_PROTOTIPOS = \"{SALIDA_CSV}\" en create_samples_celdas.ipynb")

Coincidencia de asignaciones tras releer el CSV: 100.000000%
Para usarlos: ARCHIVO_PROTOTIPOS = "src/notebooks/preprocessing/assets/prototipos_train-completo_k70.csv" en create_samples_celdas.ipynb
